In [31]:
from huggingface_hub import notebook_login
import os
notebook_login()
repo_name='jkhyjkhy/wav2vec2-large-xlsr-sw-ASR'

In [32]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [33]:
BASE = '/content/drive/MyDrive/LR_ASR/14.0-delta-2023-06-23'
CSV = f'{BASE}/manifest_sw_14_0_delta.csv'
AUDIO_DIR = f'{BASE}/cleaned_sw_audio_14_0_delta'
# If you're using a pre-configured HuggingFace processor, you can comment out or remove VOCAB
VOCAB = f'{BASE}/vocab.json'

In [34]:
import pandas as pd
import numpy as np
df = pd.read_csv(CSV)
# df['wav_path'] = df['wav_path'].apply(lambda p: os.path.join(AUDIO_DIR, p.split('\\')[-1]))
df["wav_path"] = df["wav_path"].apply(lambda p: os.path.join(AUDIO_DIR, os.path.basename(p)))
print(f"Total samples in dataset: {len(df)}")

# 실제 존재하는 파일만 필터링
df['file_exists'] = df['wav_path'].apply(os.path.exists)
print(f"원본 샘플 수: {len(df)}")
print(f"없는 파일 수: {len(df[~df['file_exists']])}")

df = df[df['file_exists']].drop(columns=['file_exists'])
print(f"필터링 후 샘플 수: {len(df)}")


Total samples in dataset: 270
원본 샘플 수: 270
없는 파일 수: 61
필터링 후 샘플 수: 209


In [35]:
from datasets import Dataset, Audio
dataset = Dataset.from_pandas(df[['wav_path', 'transcript']])

In [36]:
import re

def remove_special_characters(batch, column_names='transcript'):
  chars_to_ignore_regex = '[\,\?\.\!\-\;\:\"\(\)]'
  batch[column_names] = re.sub(chars_to_ignore_regex, '', batch[column_names]).lower() + " "
  return batch

dataset = dataset.map(remove_special_characters)

print(len(dataset['wav_path']))

<>:4: SyntaxWarning: invalid escape sequence '\,'
<>:4: SyntaxWarning: invalid escape sequence '\,'
/tmp/ipython-input-4221671052.py:4: SyntaxWarning: invalid escape sequence '\,'
  chars_to_ignore_regex = '[\,\?\.\!\-\;\:\"\(\)]'


Map:   0%|          | 0/209 [00:00<?, ? examples/s]

209


In [37]:
from transformers import AutoProcessor, AutoModelForCTC

processor = AutoProcessor.from_pretrained("jkhyjkhy/wav2vec2-large-xlsr-sw-ASR")
model = AutoModelForCTC.from_pretrained("jkhyjkhy/wav2vec2-large-xlsr-sw-ASR")

Loading weights:   0%|          | 0/424 [00:00<?, ?it/s]

In [38]:
import torch
def map_to_result(batch):
  with torch.no_grad():
    input_values = torch.tensor(batch["input_values"]).unsqueeze(0)
    logits = model(input_values).logits

  pred_ids = torch.argmax(logits, dim=-1)
  print(pred_ids)
  batch["pred_str"] = processor.batch_decode(pred_ids)[0]
  batch["text"] = processor.decode(batch["labels"], group_tokens=False)

  return batch

In [39]:
dataset = dataset.cast_column("wav_path", Audio(sampling_rate=16000))

def prepare_dataset(batch):
    audio = batch["wav_path"]

    audio_inputs = processor(audio["array"], sampling_rate=audio["sampling_rate"])

    batch["input_values"] = audio_inputs["input_values"][0]
    batch["input_length"] = len(batch["input_values"])
    label_inputs = processor.tokenizer(batch["transcript"])
    batch["labels"] = label_inputs["input_ids"]

    return batch

In [40]:
dataset.column_names
embeded_dataset = dataset.map(prepare_dataset, remove_columns=dataset.column_names)

Map:   0%|          | 0/209 [00:00<?, ? examples/s]

In [41]:
embeded_dataset.column_names

['input_values', 'input_length', 'labels']

In [42]:
!pip install evaluate jiwer bert_score sacrebleu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 79.7 MB/s eta 0:00:00


In [44]:
import evaluate

# Load metrics
wer_metric = evaluate.load('wer')
cer_metric = evaluate.load('cer')
bleu_metric = evaluate.load('bleu')
bertscore_metric = evaluate.load('bertscore')

# SER (Sentence Error Rate) - custom function
def compute_ser(predictions, references):
    """Calculate Sentence Error Rate - percentage of sentences with any error"""
    errors = sum(1 for p, r in zip(predictions, references) if p.strip() != r.strip())
    return errors / len(references)

# Levenshtein Distance - custom function
def levenshtein_distance(s1, s2):
    """Calculate Levenshtein distance between two strings"""
    if len(s1) < len(s2):
        return levenshtein_distance(s2, s1)
    if len(s2) == 0:
        return len(s1)

    previous_row = range(len(s2) + 1)
    for i, c1 in enumerate(s1):
        current_row = [i + 1]
        for j, c2 in enumerate(s2):
            insertions = previous_row[j + 1] + 1
            deletions = current_row[j] + 1
            substitutions = previous_row[j] + (c1 != c2)
            current_row.append(min(insertions, deletions, substitutions))
        previous_row = current_row
    return previous_row[-1]

def compute_avg_levenshtein(predictions, references):
    """Calculate average Levenshtein distance across all samples"""
    distances = [levenshtein_distance(p, r) for p, r in zip(predictions, references)]
    return np.mean(distances), distances


In [45]:
def compute_metrics(pred):
    pred_logits = pred.predictions
    pred_ids = np.argmax(pred_logits, axis=-1)

    pred.label_ids[pred.label_ids == -100] = processor.tokenizer.pad_token_id

    pred_str = processor.batch_decode(pred_ids)
    label_str = processor.batch_decode(pred.label_ids, group_tokens=False)

    # Basic metrics
    wer = wer_metric.compute(predictions=pred_str, references=label_str)
    cer = cer_metric.compute(predictions=pred_str, references=label_str)
    ser = compute_ser(pred_str, label_str)

    # Levenshtein
    avg_lev, _ = compute_avg_levenshtein(pred_str, label_str)

    return {
        "wer": wer,
        "cer": cer,
        "ser": ser,
        "levenshtein": avg_lev
    }

In [47]:
results = embeded_dataset.map(map_to_result, remove_columns=embeded_dataset.column_names)

predictions = results["pred_str"]
references = results["text"]

# 1. WER (Word Error Rate)
wer_score = wer_metric.compute(predictions=predictions, references=references)

# 2. CER (Character Error Rate)
cer_score = cer_metric.compute(predictions=predictions, references=references)

# 3. SER (Sentence Error Rate)
ser_score = compute_ser(predictions, references)

# 4. BLEU Score
# BLEU expects: predictions as strings, references as list of strings
bleu_refs = [[ref] for ref in references]  # each reference wrapped in a list
bleu_score = bleu_metric.compute(predictions=predictions, references=bleu_refs)

# 5. BERTScore
bertscore_results = bertscore_metric.compute(
    predictions=predictions,
    references=references,
    lang="sw"  # Swahili
)
bertscore_precision = np.mean(bertscore_results['precision'])
bertscore_recall = np.mean(bertscore_results['recall'])
bertscore_f1 = np.mean(bertscore_results['f1'])

# 6. Levenshtein Distance
avg_levenshtein, all_levenshtein = compute_avg_levenshtein(predictions, references)

# Print Results
print("="*60)
print("                    Evaluation Results")
print("="*60)
print(f"\n[Edit Distance Based Metrics]")
print(f"  WER (Word Error Rate):        {wer_score:.4f} ({wer_score*100:.2f}%)")
print(f"  CER (Character Error Rate):   {cer_score:.4f} ({cer_score*100:.2f}%)")
print(f"  SER (Sentence Error Rate):    {ser_score:.4f} ({ser_score*100:.2f}%)")
print(f"  Avg Levenshtein Distance:     {avg_levenshtein:.2f} chars")

print(f"\n[N-gram Based Metrics]")
print(f"  BLEU Score:                   {bleu_score['bleu']:.4f} ({bleu_score['bleu']*100:.2f}%)")

print(f"\n[Semantic Similarity Metrics]")
print(f"  BERTScore Precision:          {bertscore_precision:.4f}")
print(f"  BERTScore Recall:             {bertscore_recall:.4f}")
print(f"  BERTScore F1:                 {bertscore_f1:.4f}")

print("="*60)


Map:   0%|          | 0/209 [00:00<?, ? examples/s]

tensor([[51, 51, 51, 51, 51, 51, 51, 12, 51,  2, 51, 51, 51, 51, 51, 14, 15, 51,
         35, 48, 48, 51, 31, 31,  2, 51, 51, 51, 51, 51, 51, 51, 51, 40, 38, 51,
         48, 51, 51, 51, 51, 51, 51, 51, 51, 14, 35, 35, 48, 51, 51, 31, 51,  2,
         48, 51, 51, 51, 51, 51, 14, 51, 51, 51, 43, 51,  2, 51, 51, 51, 51, 51,
         51, 51, 51, 51, 14, 51, 33, 51, 48, 48, 51, 51, 51, 51, 51, 51, 51, 40,
         21, 21,  2, 48, 48, 51, 51, 19, 51, 20, 51, 51, 33, 51, 51, 14, 14, 38,
         51, 51, 51, 51, 51, 51, 40, 51,  2, 51, 51, 51, 51, 51, 51, 51, 14, 51,
         33, 48, 48, 51, 51, 51, 51, 51, 51, 51, 51, 51, 51, 21,  2, 51, 48, 51,
         51, 51, 14, 51, 37, 51,  2, 51, 51, 51, 51, 51, 51, 51, 14, 35, 51, 48]])
tensor([[51, 51, 51, 51, 51,  6, 51, 51,  2, 51, 51, 51, 51, 37, 51, 35, 51, 51,
         51, 51, 51, 36, 51, 51, 35, 48, 48, 51, 51, 51, 51, 31, 51,  2, 48, 48,
         51, 21,  2, 51, 33, 33, 51, 51, 51, 51, 51, 14, 51, 15, 51, 33, 51, 51,
         51, 51, 51, 51, 5

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


                    Evaluation Results

[Edit Distance Based Metrics]
  WER (Word Error Rate):        0.3433 (34.33%)
  CER (Character Error Rate):   0.1218 (12.18%)
  SER (Sentence Error Rate):    0.8421 (84.21%)
  Avg Levenshtein Distance:     7.07 chars

[N-gram Based Metrics]
  BLEU Score:                   0.4798 (47.98%)

[Semantic Similarity Metrics]
  BERTScore Precision:          0.9046
  BERTScore Recall:             0.9033
  BERTScore F1:                 0.9038


In [48]:
from IPython.display import display, HTML
import random
def show_random_elements(dataset, num_examples=10):
    assert num_examples <= len(dataset), "Can't pick more elements than there are in the dataset."
    picks = []
    for _ in range(num_examples):
        pick = random.randint(0, len(dataset)-1)
        while pick in picks:
            pick = random.randint(0, len(dataset)-1)
        picks.append(pick)

    df = pd.DataFrame(dataset[picks])
    display(HTML(df.to_html()))

In [49]:
show_random_elements(results)

,pred_str,text
0,irani yaamua kusitisha mazungunuzo yake ya siri na saudia,iran yaamua kusitisha mazungumzo yake ya siri na saudi
1,katika juhudi za kulilenga vye makundi la halshababu na viongozi wake ya uandishi wa habari,katika juhudi za kulilenga vyema kundi la alshabaab na viongozi wake afisa mwandamizi wa utawala aliwaambia waandishi wa habari
2,marais mhuunu kanyata wa kenya yureri musaveni na uganda unaporu kugame wa rwanda,marais uhuru kenyata wa kenya yoweri museveni wa uganda na paul kagame wa rwanda
3,kumi na mbili hadi shilingi elfu mbili mia sita tisini na mbili za tanzania,kumi na mbili hadi shilingi elfu mbili mia sita tisini na mbili za tanzania
4,baadhi ya waasi ha walirudi kongo,baadhi ya waasi hao walirudi kongo
5,tizinizi waavijermani unaweza tumia jiko la gesi kupika vyazilishe,unaweza tumia jiko la gasi kupika viazi lishe
6,wanajeshi wa kitengo cha unga marekani wamepelekwa opolan,wanajeshi wa kitengo cha anga marekani wamepelekwa upoland
7,ambao walisababishwa ipo vya maelufu ya watu mwaka tisini na tisa,ambao ulisababishwa vifo vya maelfu ya watu mwaka tisini na tisa
8,machangazo ya idhaa ya kiswahili wafikie mamilioni ya wasikilizaji katika afrika mashariki na afrika ya kati pamoja na sehemu nyingine,matangazo ya idhaa ya kiswahili huwafikia mamilioni ya wasikilizaji katika afrika mashariki na afrika ya kati pamoja na sehemu nyingine
9,jinsi ya kuandaa viazilishi,jinsi ya kuanda viazi lishe
